In [1]:
import pandas as pd
import geopandas as gpd
import time
import maup
from maup import smart_repair
import os

In [2]:
# Using the 2024 draft
start_time = time.time()
# Overall
election_df = gpd.read_file(r".\pa_2024_gen_prec_draft\pa_2024_gen_prec_draft.shp")

# Saved
# election_df = gpd.read_file(r".\pa_2024_gen_prec_repaired_county.shp")
end_time = time.time()

print("The time to import pa_2024_gen_prec_draft.shp is:", (end_time-start_time)/60, "mins")

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: .\pa_2024_gen_prec_draft\pa_2024_gen_prec_draft.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


The time to import pa_2024_gen_prec_draft.shp is: 0.025083927313486735 mins


In [3]:
print(election_df.columns)

Index(['UNIQUE_ID', 'COUNTYFP', 'County', 'Precinct', 'Muni_Name', 'Muni_Type',
       'Muni_Code1', 'Muni_Name1', 'Muni_Code2', 'Muni_Name2',
       ...
       'GSU41RPIT', 'GSU43DCOS', 'GSU45DPIS', 'GSU45RDIN', 'GSU47DLEN',
       'GSU47RVOG', 'GSU49DWER', 'GSU49RLAU', 'TOT_VOTES', 'geometry'],
      dtype='object', length=429)


In [4]:
# https://redistrictingdatahub.org/dataset/pennsylvania-block-pl-94171-2020-by-table/
start_time = time.time()
population_df = gpd.read_file(r".\pa_pl2020_b\pa_pl2020_p2_b.shp")
end_time = time.time()

print("The time to import pa_pl2020_p2_b.shp is:", (end_time-start_time)/60, "mins")

The time to import pa_pl2020_p2_b.shp is: 0.1795823574066162 mins


In [5]:
population_df.columns

Index(['GEOID20', 'SUMLEV', 'LOGRECNO', 'GEOID', 'COUNTY', 'P0020001',
       'P0020002', 'P0020003', 'P0020004', 'P0020005', 'P0020006', 'P0020007',
       'P0020008', 'P0020009', 'P0020010', 'P0020011', 'P0020012', 'P0020013',
       'P0020014', 'P0020015', 'P0020016', 'P0020017', 'P0020018', 'P0020019',
       'P0020020', 'P0020021', 'P0020022', 'P0020023', 'P0020024', 'P0020025',
       'P0020026', 'P0020027', 'P0020028', 'P0020029', 'P0020030', 'P0020031',
       'P0020032', 'P0020033', 'P0020034', 'P0020035', 'P0020036', 'P0020037',
       'P0020038', 'P0020039', 'P0020040', 'P0020041', 'P0020042', 'P0020043',
       'P0020044', 'P0020045', 'P0020046', 'P0020047', 'P0020048', 'P0020049',
       'P0020050', 'P0020051', 'P0020052', 'P0020053', 'P0020054', 'P0020055',
       'P0020056', 'P0020057', 'P0020058', 'P0020059', 'P0020060', 'P0020061',
       'P0020062', 'P0020063', 'P0020064', 'P0020065', 'P0020066', 'P0020067',
       'P0020068', 'P0020069', 'P0020070', 'P0020071', 'P002

In [6]:
# https://redistrictingdatahub.org/dataset/pennsylvania-county-boundaries-2020/
start_time = time.time()
county_df = gpd.read_file(r".\pa_pl2020_cnty\pa_pl2020_cnty.shp")
end_time = time.time()

print("The time to import pa_pl2020_cnty.shp is:", (end_time-start_time)/60, "mins")

The time to import pa_pl2020_cnty.shp is: 0.0007920304934183757 mins


In [7]:
county_df.columns

Index(['STATEFP20', 'COUNTYFP20', 'COUNTYNS20', 'GEOID20', 'NAME20',
       'NAMELSAD20', 'LSAD20', 'CLASSFP20', 'MTFCC20', 'CSAFP20',
       ...
       'P0050002', 'P0050003', 'P0050004', 'P0050005', 'P0050006', 'P0050007',
       'P0050008', 'P0050009', 'P0050010', 'geometry'],
      dtype='object', length=349)

In [8]:
# https://redistrictingdatahub.org/dataset/2022-pennslyvania-congressional-districts-approved-plan/
start_time = time.time()
district_df = gpd.read_file(r".\pa_cong_adopted_2022\carter_boundaries.shp")
end_time = time.time()

print("The time to import carter_boundaries is:", (end_time-start_time)/60, "mins")

The time to import carter_boundaries is: 0.0004712382952372233 mins


In [9]:
district_df.columns

Index(['ID', 'AREA', 'DISTRICT', 'geometry'], dtype='object')

The 2024 election_df covers a lot of elections and county info but doesn't have population information and district assignments so I need to merge PL 94-171 census data and assign districts

In [10]:
# Convert to UTM
population_df = population_df.to_crs(population_df.estimate_utm_crs())
election_df = election_df.to_crs(election_df.estimate_utm_crs())
district_df = district_df.to_crs(district_df.estimate_utm_crs())
county_df = county_df.to_crs(county_df.estimate_utm_crs())

In [11]:
# maup.doctor raised a topology error on the precinct shapefile.
# I then attempted a full smart_repair pass, but it was computationally expensive
# on this dataset and did not finish in a practical amount of time.
election_df = smart_repair(election_df)

Snapping all geometries to a grid with precision 10^( -4 ) to avoid GEOS errors.
Identifying overlaps...
Resolving overlaps...
Assigning order 2 pieces...
Assigning order 3 pieces...
Couldn't find a polygon to glue a component in the intersection of geometries {1154, 51, 1156} to
Couldn't find a polygon to glue a component in the intersection of geometries {330, 331, 332} to
Couldn't find a polygon to glue a component in the intersection of geometries {938, 515, 516} to
1 gaps will remain unfilled, because they exceed the area threshold.
1 gaps will remain unfilled, because they are not simply connected.
Filling gaps...


Gaps to simplify: 6115it [5:18:48,  3.13s/it]                               
Gaps to fill: 100%|██████████| 819/819 [1:00:28<00:00,  4.43s/it]


In [12]:
maup.doctor(county_df)

True

In [13]:
# Check for issues
maup.doctor(election_df)

There are 5 holes.


False

In [14]:
# Check for issues
maup.doctor(population_df)

True

In [15]:
# Check for issues
maup.doctor(district_df)

True

In [16]:
# county_regions = election_df.dissolve(by="County")

election_df = smart_repair(
    election_df,
    nest_within_regions=county_df,
    min_rook_length=30
)

Snapping all geometries to a grid with precision 10^( -4 ) to avoid GEOS errors.


c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\assign.py:32: AssigmentWarning: Warning: Some units in the source geometry were unassigned.
  warnings.warn(


Identifying overlaps...
Resolving overlaps and filling gaps...


c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\assign.py:32: AssigmentWarning: Warning: Some units in the source geometry were unassigned.
  warnings.warn(
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to fill in region 3: 100%|██████████| 2/2 [00:00<00:00, 49.01it/s]


1 gaps in region 4 will remain unfilled, because they exceed the area threshold.


...

Gaps to fill in region 48: 100%|██████████| 67/67 [00:21<00:00,  3.18it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 50: 100%|██████████| 1/1 [00:00<00:00, 102.53it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 52: 100%|██████████| 3/3 [00:00<00:00, 126.88it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to fill in region 53: 100%|██████████| 17/17 [00:01<00:00, 13.25it/s]   
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 55: 100%|██████████| 1/1 [00:00<00:00, 94.38it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 59: 100%|██████████| 6/6 [00:00<00:00, 125.06it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 60: 100%|██████████| 32/32 [00:00<00:00, 91.15it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to fill in region 63: 100%|██████████| 13/13 [00:00<00:00, 20.47it/s]


1 gaps in region 64 will remain unfilled, because they exceed the area threshold.


Gaps to simplify in region 64: 291it [00:54,  5.31it/s]                         
Gaps to fill in region 66: 100%|██████████| 40/40 [00:01<00:00, 31.23it/s]


Converting small rook adjacencies to queen...


c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\adjacencies.py:91: IslandWarning: Found islands.
Indices of islands: {3940, 3998, 3861, 629, 3964, 3869, 3870, 3999}
  warnings.warn(


try this first instead of regular smart repair()

In [17]:
maup.doctor(election_df)

There are 2 holes.


False

In [18]:
# maup.doctor(repaired_election_county)

In [19]:
election_df.to_file("pa_2024_gen_prec_repaired_county.shp")

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Creating a 256th field, but some DBF readers might only support 255 fields
  ogr_write(


In [20]:
# election_df = gpd.read_file(r".\pa_2024_gen_prec_repaired_county.shp")
final_df = election_df.copy()

In [21]:
# Assign each census block to a precinct
blocks_to_precinct = maup.assign(population_df, final_df.geometry)

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\assign.py:32: AssigmentWarning: Warning: Some units in the source geometry were unassigned.
  warnings.warn(


In [22]:
blocks_to_precinct

0         2997.0
1         6282.0
2         6283.0
3         6263.0
4         6273.0
           ...  
336980    1963.0
336981    1912.0
336982    1949.0
336983    1969.0
336984    1961.0
Length: 336985, dtype: float64

In [23]:
pop_columns = ['P0020001', 'P0020002', 'P0020005', 'P0020006', 'P0020007', 'P0020008', 'P0020009', 'P0020010', 'P0020011']

In [24]:
# Sum the block population into each precinct
for name in pop_columns:
    final_df[name] = population_df[name].groupby(blocks_to_precinct).sum()

In [25]:
final_df.columns

Index(['UNIQUE_ID', 'COUNTYFP', 'County', 'Precinct', 'Muni_Name', 'Muni_Type',
       'Muni_Code1', 'Muni_Name1', 'Muni_Code2', 'Muni_Name2',
       ...
       'geometry', 'P0020001', 'P0020002', 'P0020005', 'P0020006', 'P0020007',
       'P0020008', 'P0020009', 'P0020010', 'P0020011'],
      dtype='object', length=438)

In [26]:
# Check that no one was lost
print(population_df['P0020001'].sum())
print(final_df['P0020001'].sum())

13002700
13002674.0


prof is okay with losing 30 people. Mention it!! also find their demographics too


In [27]:
rename = {'P0020001': 'TOTPOP', 'P0020002': 'HISP', 'P0020005': 'NH_WHITE', 'P0020006': 'NH_BLACK', 'P0020007': 'NH_AMIN', 'P0020008': 'NH_ASIAN', 'P0020009': 'NH_NHPI', 'P0020010': 'NH_OTHER', 'P0020011': 'NH_2MORE'}

In [28]:
# Finding the demographics for the people who were lost
for col, demographic in rename.items():

    original_total = population_df[col].sum()
    final_total = final_df[col].sum()

    lost = original_total - final_total

    print(f"{demographic}")
    print(f"Lost: {lost}")
    print()

TOTPOP
Lost: 26.0

HISP
Lost: 11.0

NH_WHITE
Lost: 2.0

NH_BLACK
Lost: 4.0

NH_AMIN
Lost: 3.0

NH_ASIAN
Lost: 5.0

NH_NHPI
Lost: 0.0

NH_OTHER
Lost: 0.0

NH_2MORE
Lost: 1.0



In [29]:
# Rename the column names
final_df.rename(columns = rename, inplace = True)

In [30]:
final_df.columns

Index(['UNIQUE_ID', 'COUNTYFP', 'County', 'Precinct', 'Muni_Name', 'Muni_Type',
       'Muni_Code1', 'Muni_Name1', 'Muni_Code2', 'Muni_Name2',
       ...
       'geometry', 'TOTPOP', 'HISP', 'NH_WHITE', 'NH_BLACK', 'NH_AMIN',
       'NH_ASIAN', 'NH_NHPI', 'NH_OTHER', 'NH_2MORE'],
      dtype='object', length=438)

In [31]:
keep_cols = [
    "UNIQUE_ID",
    "COUNTYFP",
    "Precinct",
    "geometry",

    # population
    "TOTPOP", "HISP", "NH_WHITE", "NH_BLACK",
    "NH_AMIN", "NH_ASIAN", "NH_NHPI",
    "NH_OTHER", "NH_2MORE",

    # elections
    "G24PREDHAR", "G24PRERTRU",
    "G24USSDCAS", "G24USSRMCC",
    "G24ATGDDEP", "G24ATGRSUN",
]

In [32]:
final_df = final_df[keep_cols]

In [33]:
final_df.columns

Index(['UNIQUE_ID', 'COUNTYFP', 'Precinct', 'geometry', 'TOTPOP', 'HISP',
       'NH_WHITE', 'NH_BLACK', 'NH_AMIN', 'NH_ASIAN', 'NH_NHPI', 'NH_OTHER',
       'NH_2MORE', 'G24PREDHAR', 'G24PRERTRU', 'G24USSDCAS', 'G24USSRMCC',
       'G24ATGDDEP', 'G24ATGRSUN'],
      dtype='object')

In [34]:
# Assigning precincts to districts
precincts_to_districts = maup.assign(final_df, district_df.geometry)
final_df["CD"] = precincts_to_districts

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\assign.py:32: AssigmentWarning: Warning: Some units in the source geometry were unassigned.
  warnings.warn(


In [35]:
precincts_to_districts

0       14.0
1       14.0
2       14.0
3       14.0
4       14.0
        ... 
9173    10.0
9174    10.0
9175    10.0
9176    10.0
9177     9.0
Length: 9178, dtype: float64

In [36]:
print(set(final_df['CD']))

{0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, nan, nan, nan, nan, nan, nan, nan, nan}


In [37]:
precincts_to_districts.isna().sum()

np.int64(8)

In [38]:
missing = final_df[precincts_to_districts.isna()]
len(missing)

8

In [39]:
missing["TOTPOP"].sum()

np.float64(0.0)

In [40]:
# Removing precinct with null values (there was no population in these precincts)
final_df = final_df[precincts_to_districts.notna()].copy()

In [41]:
# Store the assigned district index
final_df["CD"] = final_df["CD"].astype(int)

# Create a lookup from the index to the actual district number
district_lookup = district_df["DISTRICT"].reset_index(drop=True)

# Convert index into the actual district label using map
final_df["CD"] = final_df["CD"].map(district_lookup)

In [42]:
print(set(final_df["CD"]))
print(set(district_df["DISTRICT"]))
print(final_df["CD"].isna().sum())

{'2', '5', '7', '16', '9', '14', '3', '13', '11', '4', '8', '6', '15', '12', '1', '10', '17'}
{'2', '5', '7', '16', '14', '3', '13', '8', '4', '17', '6', '15', '9', '1', '12', '10', '11'}
0


In [43]:
# Check for missing values in each column
cols = ["TOTPOP", "HISP", "NH_WHITE", "NH_BLACK", "NH_AMIN", "NH_ASIAN", "NH_NHPI", "NH_OTHER", "NH_2MORE"]

for c in final_df[cols]:
    print(f"{c}: {final_df[c].isna().sum()} missing values")

TOTPOP: 3 missing values
HISP: 3 missing values
NH_WHITE: 3 missing values
NH_BLACK: 3 missing values
NH_AMIN: 3 missing values
NH_ASIAN: 3 missing values
NH_NHPI: 3 missing values
NH_OTHER: 3 missing values
NH_2MORE: 3 missing values


In [44]:
# Fill the missing values with 0
final_df[cols] = final_df[cols].fillna(0)

for c in final_df[cols]:
    print(f"{c}: {final_df[c].isna().sum()} missing values")

TOTPOP: 0 missing values
HISP: 0 missing values
NH_WHITE: 0 missing values
NH_BLACK: 0 missing values
NH_AMIN: 0 missing values
NH_ASIAN: 0 missing values
NH_NHPI: 0 missing values
NH_OTHER: 0 missing values
NH_2MORE: 0 missing values


In [45]:
# Create the shapefile
directory = "./PA"
if not os.path.exists(directory):
    os.makedirs(directory)
final_df.to_file("./PA/PA.shp")